# 3. Dimensionality reduction: PCA clustering vs KinCore <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.


## Table of contents

- [3.1 Principal component analysis (PCA)](#31)
- [3.2 PCA clustering vs KinCore](#3-2-pca-clustering-vs-kincore)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["07-DimensionalityReduction"]
  m0["workflow.pca_analysis"]
  nb --> m0
  m1["workflow.utilities"]
  nb --> m1
```


![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
import os

from workflow.pca_analysis import PCAWorkflow
from workflow.pca_analysis import ClusterAnalyzer
from workflow.utilities import PDBDownloader
from workflow.utilities import (
    count_pdb_files,
    braf_res,
    clear_and_make,
    make_seg,
    copy_filtered_pdbs,
    copy_cg_chain_small_molecules,
)


## 3.1 Principal component analysis (PCA)  <a id="31"></a>
We are now going to perform Principal Component Analysis on the fitted activation-loop coordinates.


We have written the class `PCAWorkflow()` in order to apply PCA to our activation loop dataset. PCA is run on **fitted** loops (`Results/activation_segments/fitted/`), where every structure has the same number of interpolated Cα points.


In [ ]:
from workflow.pca_analysis import PCAWorkflow

pca_workflow = PCAWorkflow(n_components=8, n_clusters=2)
pca_results = pca_workflow.run_full_analysis(
    structures_path="Results/activation_segments/fitted/",
    output_prefix="my_analysis"
)

# Access results
print(f"Structures: {len(pca_results['structure_names'])}")
print(f"PC1: {pca_results['explained_variance'][0]:.1f}% variance")


We now use the class `ClusterAnalyzer()` in order to visualise how our dataset projects along the first two principal components, coloured by cluster labels and by KinCore activation state.


In [ ]:
from workflow.pca_analysis import ClusterAnalyzer

cluster_analyzer = ClusterAnalyzer(n_clusters=2)

_ = cluster_analyzer.plot_pca_cluster_and_activation(
    pca_results,
    kincore_file="Results/dunbrack_assignments/kinase_conformation_assignments.csv",
    cluster_plot_path="pca_clustering_labels.png",
    activation_plot_path="pca_activation_states.png",
    show=True
)


## 3.2 PCA clustering vs KinCore <a id="3-2-pca-clustering-vs-kincore"></a>


Let's now investigate whether there is a correlation between the labels assigned by Dunbrack and the ones obtained through clustering in PC space.


In [ ]:
import os
from workflow.pca_analysis import ClusterAnalyzer

cluster_analyzer = ClusterAnalyzer(n_clusters=2)

pca_labels_file = "cluster_labels_my_analysis_hierarchical.txt"
if not os.path.exists(pca_labels_file):
    raise FileNotFoundError(
        f"Missing PCA labels file: {pca_labels_file}. Re-run the PCA cell first."
    )

os.makedirs("Results/dunbrack_assignments", exist_ok=True)

print("=== Dunbrack merge ===")
merged_out = cluster_analyzer.integrate_dunbrack_with_pca_clusters(
    pca_labels_file=pca_labels_file,
    dunbrack_assignments_csv="Results/dunbrack_assignments/kinase_conformation_assignments.csv",
    prefix_len=6,
    merged_output_csv="Results/dunbrack_assignments/pca_dunbrack_merged.csv",
    print_tables=True,
    print_percentages=True,
)

merged = merged_out["merged"]


In [ ]:
print("=== Cluster vs activity ===")
cluster_vs_activity_out = cluster_analyzer.analyze_cluster_vs_activity_status(
    merged_csv="Results/dunbrack_assignments/pca_dunbrack_merged.csv",
    print_tables=True,
    print_percentages=True,
    print_enrichment=True,
    enrichment_threshold_pct=10.0,
)

merged = cluster_vs_activity_out["merged"]
merged_clean = cluster_vs_activity_out["merged_clean"]
activity_crosstab = cluster_vs_activity_out["activity_crosstab"]


It can be useful to visualise if there is a correlation with a confusion matrix.


In [ ]:
out = cluster_analyzer.plot_cluster_vs_activation_state_heatmap(
    merged_csv="Results/dunbrack_assignments/pca_dunbrack_merged.csv",
    output_png="Results/dunbrack_assignments/pca_activation_correlation_plot.png",
    show=True,
)

merged = out["merged"]
merged_clean = out["merged_clean"]
activation_crosstab = out["activation_crosstab"]


In [ ]:
out = cluster_analyzer.plot_cluster_vs_activation_state_grouped_bar(
    merged_csv="Results/dunbrack_assignments/pca_dunbrack_merged.csv",
    output_png="Results/dunbrack_assignments/pca_activation_grouped_bar.png",
    show=True,
)
